# Does the tool run for you? Fit a planet with Hipparcos and Gaia in orbitize!

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/gaia_01_first_orbitize_fit.ipynb)

The lab's Gaia project fits a star's velocities and its motion across the sky together, to find the true mass and tilt of companions that velocities alone cannot pin down. The code is orbitize!, a public package with its own tutorials ([Blunt et al. 2020](https://arxiv.org/abs/1910.01756)). In this notebook you install it, run its Hipparcos and Gaia example as written, run it again for longer, and decide whether the longer run has settled.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. The last cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# orbitize! is built from source in Colab, so this takes about two minutes the first time.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None or importlib.util.find_spec("orbitize") is None:
    %pip install -q "habtrain[orbits] @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/gaia_01_first_orbitize_fit.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## Why velocities alone are not enough

A planet pulls its star round in a small orbit. Velocities measure only the part of that motion towards and away from us. If the orbit is tilted, part of the motion is across the sky, which velocities cannot see. So a velocity fit gives the planet's mass times the sine of the orbit's tilt (its **inclination**, 90 degrees when we see the orbit edge on). That is the **minimum mass**: the true mass is at least this, and more if the orbit is tilted.

The star's motion across the sky breaks that. Two space telescopes measured it:
- **Hipparcos** measured the positions and motions of about 118,000 stars around 1991.
- **Gaia** measured them again, much more precisely, around 2016.

A star with no companion moves across the sky in a straight line at a steady speed. A star pulled by a companion curves. So the motion Hipparcos saw, the motion Gaia saw, and the average motion over the 25 years between them (from the change in position) disagree. The **Hipparcos-Gaia Catalog of Accelerations** ([Brandt 2021](https://arxiv.org/abs/2105.11662)) lists those three motions for 115,000 stars. Fitting them with the velocities gives the true mass and the tilt.

## The example, as written

This is orbitize!'s own example: beta Pictoris b, a giant planet seen directly in images, with the star's motions from the catalog. All four files it needs ship inside orbitize!, so nothing downloads. The example is set to run fast, not to give a result: 50 walkers, 100 steps each, and 1 step of burn-in (the steps thrown away while the walkers find the right region). It takes under a minute.

In [ ]:
import os, time, warnings
import astropy.units as u

# orbitize! warns when a walker tries a value outside the allowed range; that is expected, so keep the output quiet.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="orbitize")
from orbitize import DATADIR, hipparcos, gaia, read_input, system, priors, sampler
import orbitize
print("orbitize! version", orbitize.__version__)

# The star's motions from the catalog, with the Hipparcos record and the times Gaia looked at it.
t0 = time.time()
hip = hipparcos.HipparcosLogProb(os.path.join(DATADIR, "HIP027321.d"), 27321, 1)
hgca = gaia.HGCALogProb(27321, hip, os.path.join(DATADIR, "gaia_edr3_betpic_epochs.csv"))

# The planet's positions in images, and the star's mass (solar masses) and parallax (milliarcseconds).
images = read_input.read_file(os.path.join(DATADIR, "betaPic.csv"))
beta_pic = system.System(1, images, 1.75, 51.44, mass_err=0.05, plx_err=0.12, fit_secondary_mass=True, gaia=hgca)

# Allow the planet's mass anywhere from 1 to 50 Jupiter masses, evenly in its logarithm.
mjup = u.Mjup.to(u.Msun)
beta_pic.sys_priors[beta_pic.param_idx["m1"]] = priors.LogUniformPrior(mjup, 50 * mjup)
setup_seconds = time.time() - t0

# The example's settings: 2 temperatures, 50 walkers, 100 steps each, 1 step of burn-in.
t1 = time.time()
demo = sampler.MCMC(beta_pic, 2, 50, 1)
demo.run_sampler(100 * 50, burn_steps=1, thin=10)
demo_seconds = time.time() - t1
print(f"Setup {setup_seconds:.0f} s, sampling {demo_seconds:.0f} s")

In [ ]:
def mass_range(run):
    """The planet's mass from a run: the middle value and how far the middle 68% reaches below and above, in Jupiter masses."""
    m = run.results.post[:, run.system.param_idx["m1"]] / mjup
    lo, mid, hi = np.percentile(m, [16, 50, 84])
    return round(mid, 1), round(mid - lo, 1), round(hi - mid, 1)

demo_mass = mass_range(demo)
print(f"Demo: mass of b = {demo_mass[0]} -{demo_mass[1]} +{demo_mass[2]} Jupiter masses")

# The fitted orbits against the star's three measured motions.
fig_demo = demo.results.plot_propermotion(cbar_param="m1", alpha=0.1, periods_to_plot=3)
plt.show()

The plot shows the star's motion east-west (top) and north-south (bottom) over time. The points with error bars are the three measured motions; the coloured lines are orbits the walkers tried, coloured by the planet's mass.

**Predict first**

The demo's mass range is very wide. If you run 10 times as many steps, with a real burn-in, will the range get much narrower, a little narrower, or stay the same?

In [ ]:
# Your prediction, before the longer run.
MY_PREDICTION = ...   # "much narrower", "a little narrower", "the same" or "unsure"
ht.record.predict("longer", MY_PREDICTION)

## A longer run

Now 100 walkers, 1,000 steps of burn-in and 1,000 steps each. On one Colab core this takes about 8 to 10 minutes. Leave the tab open while it runs.

In [ ]:
# This only runs once you have made your prediction above.
if ht.record.revealed("longer"):
    t2 = time.time()
    longer = sampler.MCMC(beta_pic, 2, 100, 1)
    longer.run_sampler(1000 * 100, burn_steps=1000, thin=10)
    longer_seconds = time.time() - t2
    longer_mass = mass_range(longer)
    print(f"Sampling {longer_seconds / 60:.1f} minutes")
    print(f"Longer run: mass of b = {longer_mass[0]} -{longer_mass[1]} +{longer_mass[2]} Jupiter masses")

## Has it settled?

A run has settled when the walkers stop drifting: each one wanders around the same band, step after step, with no trend. The plot below shows every walker's mass at each saved step of the longer run (one saved step for every 10 run). The mass axis is in solar masses: multiply by 1,047 for Jupiter masses, so 0.01 is about 10. Look for a trend over the steps, and for groups of walkers stuck apart from the rest.

For comparison, a published fit of the same planet, with both of the star's planets, more data and much longer runs, gives 9.3 (+2.6, -2.5) Jupiter masses for b ([Brandt et al. 2021](https://arxiv.org/abs/2011.06215)). This example fits one planet, so it should not match closely.

In [ ]:
# Each walker's mass of b at every saved step of the longer run.
if "longer" in dir():
    fig_chains = longer.examine_chains(param_list=["m1"], transparency=0.3)[0]
    plt.show()

> **Write down: your two runs**
>
> - The orbitize! version, and how long each run's sampling took. Put them in `RUN_TIMES`, for example "3.4.0; demo 40 s; longer 9 min".
> - Did the longer run settle, and how can you tell from the walker plot? One or two lines. Put it in `SETTLED`.
> - Why is the demo's mass not a result you could report? One line. Put it in `DEMO_NOT_RESULT`.
>
> **Where to look:** the printed times and masses, and the walker plot.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

A settled run looks like one flat band across the plot. A run that has not settled shows a slope, or two or more bands that never join.

</details>

In [ ]:
RUN_TIMES = ...         # one line, in quotes
SETTLED = ...           # one or two lines, in quotes
DEMO_NOT_RESULT = ...   # one line, in quotes

# Keep the answers, with both masses the runs gave, for your hand-in table.
ht.record.save("run_times", RUN_TIMES, "orbitize! version and sampling times")
ht.record.save("demo_mass", dict(zip(["mass", "minus", "plus"], demo_mass)), "the demo's mass of b (Jupiter masses)")
if "longer_mass" in dir():
    ht.record.save("longer_mass", dict(zip(["mass", "minus", "plus"], longer_mass)), "the longer run's mass of b (Jupiter masses)")
ht.record.save("settled", SETTLED, "whether the longer run settled, and how you can tell")
ht.record.save("demo_not_result", DEMO_NOT_RESULT, "why the demo's mass is not a result")

### Hand in

Run the cell below. It shows all your answers in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `orbitize-runs`, and download them.

In [ ]:
# Your answers and both plots, saved as your hand-in.
figs = {"proper_motion_demo": fig_demo}
if "fig_chains" in dir():
    figs["walkers"] = fig_chains
ht.record.handin("orbitize-runs", ["run_times", "demo_mass", "longer_mass", "settled", "demo_not_result"], figures=figs)